# Smart Posture Corrector

Tugas Besar IF4051 IOT

- Tazkia Nizami - 13522032
- Bagas Sambega Rosyada - 13522071
- Raden Francisco Trianto Bratadiningrat - 13522091


## Import Libraries

In [1]:
%pip install pandas numpy scipy scikit-learn matplotlib requests


[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [2]:
import os
import requests
import zipfile
import io
import re

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

## Datasets

In [3]:
def extract_nested_zip(extract_to):
    found_zip = True
    while found_zip:
        found_zip = False
        for root, dirs, files in os.walk(extract_to):
            for file in files:
                if file.lower().endswith('.zip'):
                    file_path = os.path.join(root, file)
                    folder_name = os.path.splitext(file_path)[0]
                    try:
                        with zipfile.ZipFile(file_path, 'r') as z:
                            z.extractall(folder_name)
                        print(f"  -> Extracted child: {file}")
                        os.remove(file_path)
                        found_zip = True
                    except zipfile.BadZipFile:
                        print(f"  -> Warning: Skipping invalid child zip {file}")
            if found_zip: break

def download_dataset(url, folder_name):
    base_dir = "datasets"
    target_path = os.path.join(base_dir, folder_name)
    os.makedirs(target_path, exist_ok=True)

    temp_zip = os.path.join(target_path, "temp_download.zip")

    try:
        print(f"Downloading from {url}...")
        with requests.get(url, timeout=60, stream=True) as r:
            r.raise_for_status()
            
            content_type = r.headers.get('Content-Type', '')
            is_zip = 'zip' in content_type or 'files-archive' in url or url.lower().endswith('.zip')

            with open(temp_zip, 'wb') as f:
                for chunk in r.iter_content(chunk_size=8192):
                    if chunk: 
                        f.write(chunk)

        if is_zip:
            print("Extracting main archive...")
            with zipfile.ZipFile(temp_zip, 'r') as z:
                z.extractall(target_path)
            os.remove(temp_zip) 
            
            extract_nested_zip(target_path)
            print(f"Success! All layers extracted to: {target_path}\n")
        else:
            cd = r.headers.get('Content-Disposition')
            filename = re.findall('filename="?([^";]+)"?', cd)[0] if cd and 'filename=' in cd else "data_file"
            final_path = os.path.join(target_path, filename)
            os.rename(temp_zip, final_path)
            print(f"Success! File saved as: {final_path}\n")
            
    except Exception as e:
        if os.path.exists(temp_zip): os.remove(temp_zip)
        print(f"Failed: {e}\n")


### 1) MPU6050 Gyroscope Data for Posture Tracking and Analysis   
Sumber: https://data.mendeley.com/datasets/ftn4rjnd6x/1  


In [4]:
if not os.path.exists("datasets/MPU6050 Gyroscope Data for Posture Tracking and Analysis"):
    download_dataset(
        url="https://data.mendeley.com/public-api/zip/ftn4rjnd6x/download/1", 
        folder_name=""
    )

In [5]:
df_original_dt1 = pd.read_csv('datasets/MPU6050 Gyroscope Data for Posture Tracking and Analysis/gyro_angles.csv')
df_original_dt1.head()

,timestamp,angle_x,angle_y,angle_z
0,55.514925,-0.007327,-0.007377,0.001637
1,55.585439,0.001708,-0.019853,0.021417
2,55.666026,-0.001325,-0.013913,0.060491
3,55.736540,-0.144643,0.031181,0.098260
4,55.807054,-0.316625,0.133238,0.168970


### 2) A Comprehensive Dataset with Physiological and Inertial Sensors   
Sumber: https://data.mendeley.com/datasets/phb9y6cp5c/1


In [6]:
if not os.path.exists("datasets/PIF Dataset A Comprehensive Dataset with Physiological and Inertial Features of the Human Body"):
    download_dataset(
        url="https://data.mendeley.com/public-api/zip/phb9y6cp5c/download/1", 
        folder_name=""
    )

In [7]:
df_original_dt2 = pd.read_csv('datasets/PIF Dataset A Comprehensive Dataset with Physiological and Inertial Features of the Human Body/PIF Dataset/accro_all_sub.csv')
df_original_dt2.head()

,Datetime,PID,TEMP,AccX,AccY,AccZ,GyRo X,GyRo Y,GyRo Z,AccANG X,AccANG Y,GyRoANG X,GyRoANG Y,GyRoANG Z,GetANG X,GetANG Y,GetANG Z,Label
0,12:50:18 AM,1,33.26,1.01,0.04,0.14,-0.69,0.54,0.24,1.99,-79.99,-5.13,4.01,1.76,-3.08,-75.62,1.76,Standing
1,12:50:20 AM,1,33.21,1.01,0.04,0.14,-0.87,0.57,0.46,2.11,-79.75,-4.77,3.82,2.36,-1.80,-76.42,2.36,Standing
2,12:50:21 AM,1,33.40,1.00,0.04,0.13,1.64,0.25,0.69,2.09,-80.02,-4.09,3.82,2.83,-0.44,-76.95,2.83,Standing
3,12:50:22 AM,1,33.21,1.00,0.05,0.15,2.10,-1.06,0.37,2.55,-78.41,-4.02,3.68,2.90,0.19,-77.41,2.90,Standing
4,12:50:23 AM,1,33.21,1.00,0.06,0.15,-2.51,-0.38,0.59,3.12,-78.07,-4.13,3.42,3.28,0.50,-77.83,3.28,Standing


### 3) Dataset Human Posture Recognition MPU6050  
Source: Github https://github.com/pkmandke/imu.dat 

In [8]:
if not os.path.exists("datasets/Dataset Human Posture Recognition MPU6050"):    
    download_dataset(
        url="https://drive.google.com/uc?export=download&id=1GtNRi9CgHOlGpsa-Dlp6uzAqXUVdEh54", 
        folder_name="Dataset Human Posture Recognition MPU6050"
    )


In [9]:
df_original_dt3 = pd.read_csv('datasets/Dataset Human Posture Recognition MPU6050/final_data1.csv')
df_original_dt3.head()

,index,Ax1,Ay1,Az1,Ax2,Ay2,Az2,label
0,0,-0.34741,-0.39551,0.94263,-0.05273,0.30005,0.91821,0.0
1,1,-0.13452,-0.36768,0.97095,-0.04932,0.30176,0.92676,0.0
2,2,-0.11279,-0.36182,0.96289,-0.04785,0.29639,0.92505,0.0
3,3,-0.12915,-0.36792,0.95898,-0.05713,0.29785,0.92041,0.0
4,4,-0.13379,-0.36548,0.96753,-0.05810,0.31104,0.92578,0.0


### 4) Daily Activities Wearable Dataset for Cardiorespiratory Fitness Estimation  
Sumber: https://zenodo.org/records/15857137  


In [10]:
# 500 MB
if not os.path.exists("datasets/Daily Activities Wearable Dataset for Cardiorespiratory Fitness Estimation"):
    download_dataset(
        url="https://zenodo.org/api/records/15857137/files-archive", 
        folder_name="Daily Activities Wearable Dataset for Cardiorespiratory Fitness Estimation"
    )

In [11]:
def get_dt4():
    data_root = r"datasets/Daily Activities Wearable Dataset for Cardiorespiratory Fitness Estimation/DatasetIMUandBIOMARKERS" 

    def downsample_median_all_cols(df, factor=100):
        df_numeric = df.select_dtypes(include=[np.number])
        n = len(df_numeric)
        trim = n - (n % factor)
        reshaped = df_numeric.iloc[:trim].values.reshape(-1, factor, df_numeric.shape[1])
        downsampled_values = np.median(reshaped, axis=1)
        return pd.DataFrame(downsampled_values, columns=df_numeric.columns)

    all_subjects_list = []

    for i in range(1, 68): 
        subject_id = f"Subject{i:02d}"
        folder = os.path.join(data_root, subject_id)
        bio_file = os.path.join(folder, f"BiomarkersSubject{i:02d}.csv")
        imu_file = os.path.join(folder, f"IMUSubject{i:02d}.csv")

        if not os.path.exists(bio_file) or not os.path.exists(imu_file):
            continue

        df_bio = pd.read_csv(bio_file)
        df_imu = pd.read_csv(imu_file)

        df_imu_downsampled = downsample_median_all_cols(df_imu, factor=100)

        min_len = min(len(df_imu_downsampled), len(df_bio))
        
        combined_df = pd.concat([
            df_imu_downsampled.iloc[:min_len].reset_index(drop=True),
            df_bio.iloc[:min_len].reset_index(drop=True)
        ], axis=1)
        
        combined_df['subject_id'] = subject_id
        all_subjects_list.append(combined_df)

    if not all_subjects_list:
        return pd.DataFrame()

    return pd.concat(all_subjects_list, ignore_index=True)

df_original_dt4 = get_dt4()

In [12]:
df_original_dt4.head()

,epoch,q_w_chest,q_x_chest,q_y_chest,q_z_chest,q_w_left_hand,q_x_left_hand,q_y_left_hand,q_z_left_hand,q_w_right_knee,...,a_y_right_hand,a_z_right_hand,g_x_right_hand,g_y_right_hand,g_z_right_hand,SpO2,HR,ActivityLabel,subject_id,timestamp
0,1.740000e+12,0.838,-0.004,0.546,-0.009,0.799,-0.1280,-0.394,0.436,0.993,...,0.4100,0.7410,-0.2440,-0.2440,0.732,97.0,100.0,0.0,Subject01,NaN
1,1.740000e+12,0.838,-0.007,0.546,-0.010,0.799,-0.1280,-0.394,0.436,0.993,...,0.4150,0.7430,-0.2135,-0.0915,0.671,96.0,100.0,0.0,Subject01,NaN
2,1.740000e+12,0.838,-0.008,0.545,-0.010,0.798,-0.1270,-0.397,0.436,0.993,...,0.4130,0.7440,-0.3050,-0.3050,0.732,96.0,100.0,0.0,Subject01,NaN
3,1.740000e+12,0.834,-0.006,0.551,-0.009,0.798,-0.1360,-0.396,0.436,0.994,...,0.4555,0.7205,1.2200,-0.6710,-1.463,95.0,100.0,0.0,Subject01,NaN
4,1.740000e+12,0.819,-0.002,0.573,-0.018,0.795,-0.1865,-0.384,0.430,0.993,...,0.4170,0.7200,-1.1590,-0.4880,1.341,95.0,103.0,0.0,Subject01,NaN


## Data Procesing & Cleaning

In [13]:
df_original_dt1
df_original_dt2
df_original_dt3
df_original_dt4

,epoch,q_w_chest,q_x_chest,q_y_chest,q_z_chest,q_w_left_hand,q_x_left_hand,q_y_left_hand,q_z_left_hand,q_w_right_knee,...,a_y_right_hand,a_z_right_hand,g_x_right_hand,g_y_right_hand,g_z_right_hand,SpO2,HR,ActivityLabel,subject_id,timestamp
0,1.740000e+12,0.8380,-0.004,0.5460,-0.009,0.7990,-0.1280,-0.3940,0.4360,0.993,...,0.4100,0.7410,-0.2440,-0.2440,0.7320,97.0,100.0,0.0,Subject01,NaN
1,1.740000e+12,0.8380,-0.007,0.5460,-0.010,0.7990,-0.1280,-0.3940,0.4360,0.993,...,0.4150,0.7430,-0.2135,-0.0915,0.6710,96.0,100.0,0.0,Subject01,NaN
2,1.740000e+12,0.8380,-0.008,0.5450,-0.010,0.7980,-0.1270,-0.3970,0.4360,0.993,...,0.4130,0.7440,-0.3050,-0.3050,0.7320,96.0,100.0,0.0,Subject01,NaN
3,1.740000e+12,0.8340,-0.006,0.5510,-0.009,0.7980,-0.1360,-0.3960,0.4360,0.994,...,0.4555,0.7205,1.2200,-0.6710,-1.4630,95.0,100.0,0.0,Subject01,NaN
4,1.740000e+12,0.8190,-0.002,0.5730,-0.018,0.7950,-0.1865,-0.3840,0.4300,0.993,...,0.4170,0.7200,-1.1590,-0.4880,1.3410,95.0,103.0,0.0,Subject01,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
61962,NaN,0.6965,-0.285,0.3690,0.547,0.8080,-0.3440,-0.4240,-0.2210,0.822,...,0.2910,0.3250,-0.2440,-0.1220,0.5795,97.0,101.0,0.0,Subject67,1.746721e+12
61963,NaN,0.7070,-0.276,0.3575,0.545,0.8080,-0.3450,-0.4230,-0.2220,0.822,...,0.2940,0.3230,-0.0610,-0.4880,0.5490,97.0,101.0,0.0,Subject67,1.746721e+12
61964,NaN,0.7090,-0.278,0.3540,0.543,0.8120,-0.2965,-0.2585,-0.3950,0.822,...,0.2940,0.3230,-0.0610,-0.4880,0.5490,97.0,100.0,0.0,Subject67,1.746721e+12
61965,NaN,0.7090,-0.275,0.3560,0.543,0.8115,-0.2790,-0.0830,-0.5105,0.822,...,0.2940,0.3230,-0.0610,-0.4880,0.5490,97.0,99.0,0.0,Subject67,1.746721e+12


## Model Training